In [1]:
import os
import json
from tqdm import tqdm
from modelscope import snapshot_download
from vllm import LLM, SamplingParams

In [2]:
model_id = "qwen/Qwen2.5-1.5B-Instruct"
local_model_path = snapshot_download(model_id)
print(f"Model downloaded to: {local_model_path}")

2026-03-03 12:05:32,001 - modelscope - INFO - Target directory already exists, skipping creation.


Model downloaded to: /root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct


In [2]:
# 加载模型（使用上面下载的本地路径）
llm = LLM(
    model=local_model_path,
    tokenizer=local_model_path,
    dtype="bfloat16",      # Qwen2.5 支持 bfloat16，若显存小可改 "float16"
    tensor_parallel_size=1,  # 单卡
    trust_remote_code=True   # 必须开启，因为 Qwen 使用自定义 modeling
)

INFO 03-03 11:47:24 [utils.py:223] non-default args: {'tokenizer': '/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct', 'trust_remote_code': True, 'dtype': 'bfloat16', 'disable_log_stats': True, 'model': '/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-03 11:47:30 [model.py:529] Resolved architecture: Qwen2ForCausalLM
INFO 03-03 11:47:30 [model.py:1549] Using max model len 32768
INFO 03-03 11:47:30 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-03 11:47:30 [vllm.py:689] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:30 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct', speculative_config=None, tokenizer='/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_conf

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=1964) INFO 03-03 11:47:45 [default_loader.py:293] Loading weights took 0.49 seconds
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:46 [gpu_model_runner.py:4221] Model loading took 2.89 GiB memory and 14.222162 seconds
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:49 [backends.py:916] Using cache directory: /root/.cache/vllm/torch_compile_cache/33baad2b73/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:49 [backends.py:976] Dynamo bytecode transform time: 3.12 s
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:53 [backends.py:351] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:56 [backends.py:368] Compiling a graph for compile range (1, 8192) takes 5.97 s
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:56 [monitor.py:34] torch.compile takes 9.10 s in total
(EngineCore_DP0 pid=1964) INFO 03-03 11:47:57 [gpu_worker.py:373] Available KV cache memory: 23.84 GiB
(EngineCore_DP0 pid=1964) INFO 03-03 11

(EngineCore_DP0 pid=1964) 2026-03-03 11:47:57,633 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=1964) 2026-03-03 11:47:57,651 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 46.32it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [01:33<00:00,  2.66s/it]


(EngineCore_DP0 pid=1964) INFO 03-03 11:49:32 [gpu_model_runner.py:5246] Graph capturing finished in 95 secs, took -0.35 GiB
(EngineCore_DP0 pid=1964) INFO 03-03 11:49:32 [core.py:278] init engine (profile, create kv cache, warmup model) took 106.26 seconds
INFO 03-03 11:49:33 [llm.py:355] Supported tasks: ['generate']


In [3]:
# 设置生成参数
sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.8,
    repetition_penalty=1.05,
    max_tokens=512
)

# 构造对话（Qwen2.5-Instruct 使用 chat template）
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "你好，请介绍一下你自己。"}
]

# 使用 tokenizer 的 apply_chat_template 自动格式化（vLLM 内部会调用）
prompt = llm.get_tokenizer().apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 生成
outputs = llm.generate([prompt], sampling_params)

# 输出结果
for output in outputs:
    print("Response:", output.outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Response: 您好！我是来自阿里云的大规模语言模型，我叫通义千问。我可以回答各种各样的问题，提供帮助和娱乐。如果您有任何问题或需要帮助，请随时告诉我，我会尽力提供支持。


In [4]:
def generate_multiple_responses(
    data_path,
    tokenizer,
    model,
    save_path,
    num_samples=4,           # 每条指令生成多少个回答
    max_new_tokens=2048,
    device="cuda",
    temperature=0.7,         # 控制多样性（>0 才有多样性）
    top_p=0.9,
    seed=42                  # 可复现
):
    """
    构造偏好数据：对每条指令生成多个不同回答。
    
    输出格式：
    [
      {
        "instruction": "...",
        "input": "...",
        "ground_truth": "...",
        "responses": ["resp1", "resp2", "resp3", "resp4"]  # 多个回答
      },
      ...
    ]
    """
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # 加载前 N 条数据（这里取全部，你可以在调用时传入切片）
    with open(data_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    
    results = []
    model.eval()

    with torch.no_grad():
        for item in tqdm(eval_data, desc=f"Generating {num_samples} responses per prompt"):
            messages = [
                {"role": "user", "content": item["instruction"] + ("\n" + item["input"] if item["input"].strip() else "")}
            ]
            
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
            
            inputs = tokenizer(
                prompt,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=2048
            ).to(device)
            
            responses = []
            for i in range(num_samples):
                # 关键：开启 do_sample 并设置 temperature/top_p
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,          # 必须为 True 才有多样性
                    temperature=temperature,
                    top_p=top_p,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    # 防止重复
                    repetition_penalty=1.1
                )
                
                generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
                response = tokenizer.decode(generated_ids, skip_special_tokens=True)
                responses.append(response.strip())
            
            result_item = {
                "instruction": item["instruction"],
                "input": item["input"],
                "ground_truth": item.get("output", ""),
                "responses": responses  # 注意这里是列表
            }
            results.append(result_item)
    
    # 保存
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 候选回答生成完成！共 {len(results)} 条指令，每条 {num_samples} 个回答")
    print(f"📁 已保存至: {save_path}")

In [4]:
def generate_multiple_responses_vllm(
    data_path,
    local_model_path,
    save_path,
    num_samples=4,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.9,
    seed=42,
    dtype="bfloat16",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.4
):
    # 1. 加载模型
    llm = LLM(
        model=local_model_path,
        tokenizer=local_model_path,
        dtype=dtype,
        tensor_parallel_size=tensor_parallel_size,
        trust_remote_code=True,
        seed=seed,
        gpu_memory_utilization=gpu_memory_utilization
    )

    # 2. 获取 tokenizer（用于手动 apply chat template）
    tokenizer = llm.get_tokenizer()

    # 3. 设置采样参数
    sampling_params = SamplingParams(
        n=num_samples,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_new_tokens,
        repetition_penalty=1.1,
        seed=seed
    )

    # 4. 加载数据
    with open(data_path, "r", encoding="utf-8") as f:
        eval_data = json.load(f)

    # 5. 手动构造 prompts（使用 tokenizer.apply_chat_template）
    prompts = []
    original_items = []
    for item in eval_data:
        user_content = item["instruction"]
        if item["input"].strip():
            user_content += "\n" + item["input"]
        messages = [{"role": "user", "content": user_content}]
        
        # 关键：在这里格式化 prompt
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True  # 添加生成起始符（如 <|im_start|>assistant\n）
        )
        prompts.append(prompt)
        original_items.append(item)

    # 6. 批量生成（只传 prompts 和 sampling_params）
    print(f"🚀 开始生成：共 {len(prompts)} 条指令，每条生成 {num_samples} 个回答...")
    outputs = llm.generate(
        prompts,                # ← 现在是字符串列表
        sampling_params=sampling_params
        # ❌ 不要 use_chat_template / add_generation_prompt
    )

    # 7. 整理结果
    results = []
    for i, output in enumerate(outputs):
        item = original_items[i]
        responses = [o.text.strip() for o in output.outputs]  # output.outputs 是 n 个 SampleOutput
        results.append({
            "instruction": item["instruction"],
            "input": item["input"],
            "ground_truth": item.get("output", ""),
            "responses": responses
        })

    # 8. 保存
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    print(f"✅ 候选回答生成完成！共 {len(results)} 条指令，每条 {num_samples} 个回答")
    print(f"📁 已保存至: {save_path}")

In [ ]:
# 路径配置
preference_data_path = '/root/autodl-tmp/belle_preference.json'
grpo_output_path = '/root/autodl-tmp/belle_preference_response.json'

# 只取前 若干 条（可调）
with open(preference_data_path, "r", encoding="utf-8") as f:
    all_data = json.load(f)
    first = all_data[:]

temp_file = "/root/autodl-tmp/belle_first.json"
with open(temp_file, "w", encoding="utf-8") as f:
    json.dump(first, f, ensure_ascii=False, indent=2)

# 生成多回答
generate_multiple_responses(
    data_path=temp_file,
    tokenizer=tokenizer_sft,
    model=model_sft,
    save_path=grpo_output_path,
    num_samples=4,          # 每条生成 4 个回答
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.9,
    device="cuda"
)

In [8]:
# 路径配置
preference_data_path = '/root/autodl-tmp/belle_preference.json'
grpo_output_path = '/root/autodl-tmp/belle_preference_response.json'

# 只取前若干条（这里取全部，你可改 all_data[:N]）
with open(preference_data_path, "r", encoding="utf-8") as f:
    all_data = json.load(f)
    first = all_data[:]  # 取全部；如需取前100条：all_data[:100]

temp_file = "/root/autodl-tmp/belle_first.json"
os.makedirs(os.path.dirname(temp_file), exist_ok=True)
with open(temp_file, "w", encoding="utf-8") as f:
    json.dump(first, f, ensure_ascii=False, indent=2)


In [9]:
# 调用 vLLM 版本生成
generate_multiple_responses_vllm(
    data_path=temp_file,
    local_model_path=local_model_path,
    save_path=grpo_output_path,
    num_samples=4,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.9,
    seed=42,
    dtype="bfloat16"  # 若显存不足（<12GB），改为 "float16"
)


INFO 03-03 12:10:05 [utils.py:223] non-default args: {'tokenizer': '/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct', 'trust_remote_code': True, 'dtype': 'bfloat16', 'seed': 42, 'gpu_memory_utilization': 0.4, 'disable_log_stats': True, 'model': '/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-03 12:10:05 [model.py:529] Resolved architecture: Qwen2ForCausalLM
INFO 03-03 12:10:05 [model.py:1549] Using max model len 32768
INFO 03-03 12:10:05 [scheduler.py:224] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:05 [core.py:97] Initializing a V1 LLM engine (v0.16.0) with config: model='/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct', speculative_config=None, tokenizer='/root/.cache/modelscope/hub/models/qwen/Qwen2___5-1___5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='au

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(EngineCore_DP0 pid=4915) INFO 03-03 12:10:08 [default_loader.py:293] Loading weights took 0.46 seconds
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:09 [gpu_model_runner.py:4221] Model loading took 2.89 GiB memory and 0.852013 seconds
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:12 [backends.py:916] Using cache directory: /root/.cache/vllm/torch_compile_cache/33baad2b73/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:12 [backends.py:976] Dynamo bytecode transform time: 3.24 s
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:15 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 0.888 s
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:15 [monitor.py:34] torch.compile takes 4.13 s in total
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:15 [gpu_worker.py:373] Available KV cache memory: 8.16 GiB
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:15 [kv_cache_utils.py:1307] GPU KV cache size: 305,728 tokens
(EngineCore_DP0 pid=491

(EngineCore_DP0 pid=4915) 2026-03-03 12:10:15,851 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=4915) 2026-03-03 12:10:15,861 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore_DP0 pid=4915) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/51 [00:00<?, ?it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   8%|▊         | 4/51 [00:00<00:01, 39.74it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  18%|█▊        | 9/51 [00:00<00:00, 44.01it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  27%|██▋       | 14/51 [00:00<00:00, 45.08it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  37%|███▋      | 19/51 [00:00<00:00, 45.89it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  47%|████▋     | 24/51 [00:00<00:00, 46.71it/s]
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  57%|█████▋    | 29/51 [00:00

(EngineCore_DP0 pid=4915) INFO 03-03 12:10:18 [gpu_model_runner.py:5246] Graph capturing finished in 2 secs, took -0.35 GiB
(EngineCore_DP0 pid=4915) INFO 03-03 12:10:18 [core.py:278] init engine (profile, create kv cache, warmup model) took 8.87 seconds
INFO 03-03 12:10:18 [llm.py:355] Supported tasks: ['generate']
🚀 开始生成：共 10000 条指令，每条生成 4 个回答...


Adding requests:   0%|          | 0/10000 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/40000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/…

✅ 候选回答生成完成！共 10000 条指令，每条 4 个回答
📁 已保存至: /root/autodl-tmp/belle_preference_response.json


In [3]:
!nvidia-smi

Tue Mar  3 12:05:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5090        On  |   00000000:63:00.0 Off |                  N/A |
| 43%   25C    P8             16W /  575W |       0MiB /  32607MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----